In [1]:
import os
import re
import unicodedata
from collections import defaultdict, Counter

import pandas as pd
import numpy as np

BASE_DIR = ".."

TRAIN_DIR = os.path.join(BASE_DIR, "data", "train")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")

SOURCE1 = os.path.join(TRAIN_DIR, "train_source1.tsv")
SOURCE2 = os.path.join(TRAIN_DIR, "train_source2.tsv")
SOURCE3 = os.path.join(TRAIN_DIR, "train_source3.tsv")
GROUND_TRUTH = os.path.join(TRAIN_DIR, "train_ground_truth.tsv")

VALIDATION_IDS_FILE = os.path.join(
    OUTPUT_DIR,
    "validation_s1_ids.txt"
)

CHUNK_SIZE = 100_000

print("Validation IDs:", VALIDATION_IDS_FILE)

Validation IDs: ../output/validation_s1_ids.txt


In [2]:
with open(VALIDATION_IDS_FILE, "r", encoding="utf-8") as f:
    validation_s1_ids = {
        line.strip()
        for line in f
        if line.strip()
    }

print("Validation S1 entities:", len(validation_s1_ids))

Validation S1 entities: 220682


In [3]:
validation_s1_parts = []

for chunk in pd.read_csv(
    SOURCE1,
    sep="\t",
    dtype=str,
    chunksize=CHUNK_SIZE,
    keep_default_na=False
):
    selected = chunk[
        chunk["entity_id"].isin(validation_s1_ids)
    ]

    if len(selected) > 0:
        validation_s1_parts.append(selected)

validation_s1 = pd.concat(
    validation_s1_parts,
    ignore_index=True
)

validation_s1 = validation_s1.rename(
    columns={
        "entity_id": "s1_id",
        "business_name": "s1_name",
        "business_address": "s1_address",
        "country": "s1_country"
    }
)

print("Validation S1 rows:", len(validation_s1))

Validation S1 rows: 220682


In [4]:
def normalize_text(value):
    value = "" if value is None else str(value)

    value = unicodedata.normalize(
        "NFKC",
        value
    )

    value = value.casefold()

    value = re.sub(
        r"[^\w\s]",
        " ",
        value,
        flags=re.UNICODE
    )

    value = re.sub(
        r"\s+",
        " ",
        value,
        flags=re.UNICODE
    )

    return value.strip()

In [5]:
validation_s1["name_norm"] = (
    validation_s1["s1_name"]
    .map(normalize_text)
)

validation_s1["address_norm"] = (
    validation_s1["s1_address"]
    .map(normalize_text)
)

validation_s1.head()

,s1_id,s1_name,s1_address,s1_country,name_norm,address_norm
0,S1-785847572,Consulting Nyasa Nursing Private Limited,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",India,consulting nyasa nursing private limited,2505 tower 1 oakwood runwal greens mulund gore...
1,S1-796227187,Foot & Ankle Allied Center LLC,"1216 Preston Avenue, Charlottesville City, VA",US,foot ankle allied center llc,1216 preston avenue charlottesville city va
2,S1-982210261,Mayur Hospital of Gondal,"C/O Dwarkadhis Enterprise, C.M. Palace, Ist Fl...",India,mayur hospital of gondal,c o dwarkadhis enterprise c m palace ist floor...
3,S1-791867209,WG Harbor Worldwide P.C.,"101 Oak Street, Lohrville, IA",US,wg harbor worldwide p c,101 oak street lohrville ia
4,S1-72918323,Nguyen Avalanche LLC,"42 Cedar Circle, Crossville, TN",US,nguyen avalanche llc,42 cedar circle crossville tn


In [6]:
name_exact_map = defaultdict(list)
address_exact_map = defaultdict(list)
name_country_map = defaultdict(list)
address_country_map = defaultdict(list)

for row in validation_s1.itertuples(index=False):

    if row.name_norm:
        name_exact_map[
            row.name_norm
        ].append(row.s1_id)

        name_country_map[
            (row.s1_country, row.name_norm)
        ].append(row.s1_id)

    if row.address_norm:
        address_exact_map[
            row.address_norm
        ].append(row.s1_id)

        address_country_map[
            (row.s1_country, row.address_norm)
        ].append(row.s1_id)

print("Unique normalized names:", len(name_exact_map))
print("Unique normalized addresses:", len(address_exact_map))
print("Unique country+name keys:", len(name_country_map))
print("Unique country+address keys:", len(address_country_map))

Unique normalized names: 187550
Unique normalized addresses: 219344
Unique country+name keys: 187583
Unique country+address keys: 219344


In [7]:
validation_ground_truth = {}

for chunk in pd.read_csv(
    GROUND_TRUTH,
    sep="\t",
    dtype=str,
    chunksize=CHUNK_SIZE,
    keep_default_na=False
):
    for row in chunk.itertuples(index=False):

        s1_id = row.source1_entity_id

        if s1_id not in validation_s1_ids:
            continue

        value = row.matched_entity_ids.strip()

        if value:
            ids = {
                x.strip()
                for x in value.split(",")
                if x.strip()
            }
        else:
            ids = set()

        validation_ground_truth[s1_id] = ids

print(
    "Validation ground-truth S1 entities:",
    len(validation_ground_truth)
)

Validation ground-truth S1 entities: 220682


In [8]:
true_target_to_s1 = {}

total_true_matches = 0

for s1_id, target_ids in validation_ground_truth.items():

    total_true_matches += len(target_ids)

    for target_id in target_ids:

        if target_id in true_target_to_s1:
            raise RuntimeError(
                f"Target ID appears for multiple S1 entities: {target_id}"
            )

        true_target_to_s1[target_id] = s1_id

print("Validation true matches:", total_true_matches)
print("Unique target IDs:", len(true_target_to_s1))

Validation true matches: 764045
Unique target IDs: 764045


In [9]:
ROUTES = [
    "name_exact",
    "country_name_exact",
    "address_exact",
    "country_address_exact"
]

print(ROUTES)

['name_exact', 'country_name_exact', 'address_exact', 'country_address_exact']


In [10]:
def evaluate_blocking_source(
    source_path,
    source_name,
    name_map,
    name_country_map,
    address_map,
    address_country_map,
    true_target_to_s1,
    validation_s1_ids,
    chunk_size=100_000
):
    candidate_counts = Counter()
    
    candidate_s1_sets = {
        route: set()
        for route in ROUTES
    }

    matched_true_targets = {
        route: set()
        for route in ROUTES
    }

    total_rows = 0

    for chunk_number, chunk in enumerate(
        pd.read_csv(
            source_path,
            sep="\t",
            dtype=str,
            chunksize=chunk_size,
            keep_default_na=False
        ),
        start=1
    ):

        total_rows += len(chunk)

        for row in chunk.itertuples(index=False):

            target_id = row.entity_id
            name_norm = normalize_text(row.business_name)
            address_norm = normalize_text(row.business_address)
            country = row.country

            route_hits = {}

            if name_norm:
                route_hits["name_exact"] = (
                    name_map.get(name_norm, [])
                )

                route_hits["country_name_exact"] = (
                    name_country_map.get(
                        (country, name_norm),
                        []
                    )
                )
            else:
                route_hits["name_exact"] = []
                route_hits["country_name_exact"] = []

            if address_norm:
                route_hits["address_exact"] = (
                    address_map.get(address_norm, [])
                )

                route_hits["country_address_exact"] = (
                    address_country_map.get(
                        (country, address_norm),
                        []
                    )
                )
            else:
                route_hits["address_exact"] = []
                route_hits["country_address_exact"] = []

            for route, s1_matches in route_hits.items():

                if not s1_matches:
                    continue

                candidate_counts[route] += len(s1_matches)

                for s1_id in s1_matches:

                    candidate_s1_sets[route].add(
                        s1_id
                    )

                    # Only true target IDs matter for recall.
                    if target_id in true_target_to_s1:

                        true_s1 = true_target_to_s1[target_id]

                        if s1_id == true_s1:
                            matched_true_targets[route].add(
                                target_id
                            )

        if chunk_number % 10 == 0:
            print(
                f"{source_name}: "
                f"{total_rows:,} rows scanned"
            )

    return {
        "candidate_counts": candidate_counts,
        "candidate_s1_sets": candidate_s1_sets,
        "matched_true_targets": matched_true_targets
    }

In [11]:
print("Starting Source 2 blocking experiment...")

results_s2 = evaluate_blocking_source(
    SOURCE2,
    "S2",
    name_exact_map,
    name_country_map,
    address_exact_map,
    address_country_map,
    true_target_to_s1,
    validation_s1_ids,
    CHUNK_SIZE
)

print("Source 2 complete.")

Starting Source 2 blocking experiment...
S2: 1,000,000 rows scanned
S2: 2,000,000 rows scanned
S2: 3,000,000 rows scanned
S2: 4,000,000 rows scanned
S2: 5,000,000 rows scanned
Source 2 complete.


In [12]:
print("Starting Source 3 blocking experiment...")

results_s3 = evaluate_blocking_source(
    SOURCE3,
    "S3",
    name_exact_map,
    name_country_map,
    address_exact_map,
    address_country_map,
    true_target_to_s1,
    validation_s1_ids,
    CHUNK_SIZE
)

print("Source 3 complete.")

Starting Source 3 blocking experiment...
S3: 1,000,000 rows scanned
S3: 2,000,000 rows scanned
S3: 3,000,000 rows scanned
S3: 4,000,000 rows scanned
S3: 5,000,000 rows scanned
Source 3 complete.


In [13]:
combined_candidate_counts = Counter()

for route in ROUTES:

    combined_candidate_counts[route] = (
        results_s2["candidate_counts"][route]
        +
        results_s3["candidate_counts"][route]
    )


combined_true_hits = {}

for route in ROUTES:

    combined_true_hits[route] = (
        results_s2["matched_true_targets"][route]
        |
        results_s3["matched_true_targets"][route]
    )

In [14]:
blocking_results = []

for route in ROUTES:

    true_hits = len(
        combined_true_hits[route]
    )

    candidate_pairs = (
        combined_candidate_counts[route]
    )

    candidate_recall = (
        true_hits / total_true_matches
        if total_true_matches
        else 0.0
    )

    blocking_results.append({
        "route": route,
        "candidate_pairs": candidate_pairs,
        "true_matches_captured": true_hits,
        "total_true_matches": total_true_matches,
        "candidate_recall": candidate_recall
    })

blocking_results_df = pd.DataFrame(
    blocking_results
)

blocking_results_df

,route,candidate_pairs,true_matches_captured,total_true_matches,candidate_recall
0,name_exact,2171739,167106,764045,0.218712
1,country_name_exact,2164163,167106,764045,0.218712
2,address_exact,76438,63073,764045,0.082551
3,country_address_exact,76438,63073,764045,0.082551


In [15]:
validation_count = len(validation_s1_ids)

blocking_results_df[
    "avg_candidates_per_s1"
] = (
    blocking_results_df["candidate_pairs"]
    / validation_count
)

blocking_results_df

,route,candidate_pairs,true_matches_captured,total_true_matches,candidate_recall,avg_candidates_per_s1
0,name_exact,2171739,167106,764045,0.218712,9.841034
1,country_name_exact,2164163,167106,764045,0.218712,9.806704
2,address_exact,76438,63073,764045,0.082551,0.346372
3,country_address_exact,76438,63073,764045,0.082551,0.346372


In [19]:
s1_coverage_rows = []

for route in ROUTES:

    covered_s1 = (
        results_s2["candidate_s1_sets"][route]
        |
        results_s3["candidate_s1_sets"][route]
    )

    s1_coverage_rows.append({
        "route": route,
        "s1_with_candidates": len(covered_s1),
        "validation_s1": validation_count,
        "coverage_percentage": (
            len(covered_s1)
            / validation_count
            * 100
        )
    })

s1_coverage_df = pd.DataFrame(
    s1_coverage_rows
)

s1_coverage_df

,route,s1_with_candidates,validation_s1,coverage_percentage
0,name_exact,156868,220682,71.083278
1,country_name_exact,156830,220682,71.066059
2,address_exact,53141,220682,24.080351
3,country_address_exact,53141,220682,24.080351


In [20]:
blocking_results_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "blocking_experiment_01_results.csv"
    ),
    index=False
)

s1_coverage_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "blocking_experiment_01_s1_coverage.csv"
    ),
    index=False
)

print("Blocking experiment results saved.")

Blocking experiment results saved.


In [23]:
import pickle

baseline_state = {
    "total_true_matches": total_true_matches,

    "exact_name_true_hits": (
        results_s2["matched_true_targets"]["name_exact"]
        |
        results_s3["matched_true_targets"]["name_exact"]
    ),

    "exact_name_candidate_pairs": (
        results_s2["candidate_counts"]["name_exact"]
        +
        results_s3["candidate_counts"]["name_exact"]
    ),

    "exact_name_s1_with_candidates": (
        results_s2["candidate_s1_sets"]["name_exact"]
        |
        results_s3["candidate_s1_sets"]["name_exact"]
    ),

    "exact_address_true_hits": (
        results_s2["matched_true_targets"]["address_exact"]
        |
        results_s3["matched_true_targets"]["address_exact"]
    ),

    "exact_address_candidate_pairs": (
        results_s2["candidate_counts"]["address_exact"]
        +
        results_s3["candidate_counts"]["address_exact"]
    ),

    "exact_address_s1_with_candidates": (
        results_s2["candidate_s1_sets"]["address_exact"]
        |
        results_s3["candidate_s1_sets"]["address_exact"]
    ),
}

baseline_path = os.path.join(
    OUTPUT_DIR,
    "blocking_experiment_01_baseline.pkl"
)

with open(baseline_path, "wb") as f:
    pickle.dump(
        baseline_state,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )

print("Baseline state saved.")
print(
    "Exact-name hits:",
    len(baseline_state["exact_name_true_hits"])
)
print(
    "Exact-address hits:",
    len(baseline_state["exact_address_true_hits"])
)

Baseline state saved.
Exact-name hits: 167106
Exact-address hits: 63073


In [22]:
import os

print(
    "File exists:",
    os.path.exists(baseline_path)
)

print(
    "File size:",
    os.path.getsize(baseline_path) / (1024 * 1024),
    "MB"
)

File exists: True
File size: 4.601524353027344 MB
